In [2]:
import cdsapi

dataset = "reanalysis-era5-land"
request = {
    "variable": ["land_sea_mask"],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

client = cdsapi.Client()
client.retrieve(dataset, request, str("land-sea-mask.nc"))

2025-11-18 12:55:07,204 INFO Request ID is ec398878-df76-444b-ad0a-b9c92561ee7a
2025-11-18 12:55:07,377 INFO status has been updated to accepted
2025-11-18 12:55:21,452 INFO status has been updated to running
2025-11-18 12:55:29,209 INFO status has been updated to successful


'land-sea-mask.nc'

In [3]:
import xarray as xr
ds = xr.open_dataset("land-sea-mask.nc")
ds

<xarray.Dataset> Size: 52MB
Dimensions:    (time: 1, latitude: 1801, longitude: 3600)
Coordinates:
  * time       (time) datetime64[ns] 8B 2013-11-29
  * latitude   (latitude) float32 7kB 90.0 89.9 89.8 89.7 ... -89.8 -89.9 -90.0
  * longitude  (longitude) float32 14kB 0.0 0.1 0.2 0.3 ... 359.7 359.8 359.9
Data variables:
    lsm        (time, latitude, longitude) float64 52MB ...
Attributes:
    Conventions:               CF-1.6
    history:                   Fri Jun 12 14:44:03 2020: ncpdq -U lsm_1279l4_...
    NCO:                       4.7.2
    nco_openmp_thread_number:  1

In [5]:
ds.info()

xarray.Dataset {
dimensions:
	time = 1 ;
	latitude = 1801 ;
	longitude = 3600 ;

variables:
	float64 lsm(time, latitude, longitude) ;
		lsm:units = (0 - 1) ;
		lsm:long_name = Land-sea mask ;
		lsm:standard_name = land_binary_mask ;
	float32 longitude(longitude) ;
		longitude:units = degrees_east ;
		longitude:long_name = longitude ;
	float32 latitude(latitude) ;
		latitude:units = degrees_north ;
		latitude:long_name = latitude ;
	datetime64[ns] time(time) ;
		time:long_name = time ;

// global attributes:
	:Conventions = CF-1.6 ;
	:history = Fri Jun 12 14:44:03 2020: ncpdq -U lsm_1279l4_0.1x0.1.grb_v4.nc lsm_1279l4_0.1x0.1.grb_v4_unpack.nc
2020-06-12 13:29:35 GMT by grib_to_netcdf-2.6.0: grib_to_netcdf -k 4 -o lsm_1279l4_0.1x0.1.grb_v4.nc lsm_1279l4_0.1x0.1.grb ;
	:NCO = 4.7.2 ;
	:nco_openmp_thread_number = 1 ;
}

# LSM has a resolution of 0.1 degrees

In [6]:
ds.latitude.values

array([ 90. ,  89.9,  89.8, ..., -89.8, -89.9, -90. ],
      shape=(1801,), dtype=float32)

In [8]:
ds.values

<bound method Mapping.values of <xarray.Dataset> Size: 52MB
Dimensions:    (time: 1, latitude: 1801, longitude: 3600)
Coordinates:
  * time       (time) datetime64[ns] 8B 2013-11-29
  * latitude   (latitude) float32 7kB 90.0 89.9 89.8 89.7 ... -89.8 -89.9 -90.0
  * longitude  (longitude) float32 14kB 0.0 0.1 0.2 0.3 ... 359.7 359.8 359.9
Data variables:
    lsm        (time, latitude, longitude) float64 52MB ...
Attributes:
    Conventions:               CF-1.6
    history:                   Fri Jun 12 14:44:03 2020: ncpdq -U lsm_1279l4_...
    NCO:                       4.7.2
    nco_openmp_thread_number:  1>

In [9]:
# Select the lsm DataArray and access its underlying NumPy array
all_lsm_values = ds['lsm'].values

# To inspect a small portion (e.g., the top-left corner):
print(all_lsm_values[:5, :5])

[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]


In [10]:
# The coordinates for ERA5-Land are typically named 'latitude' and 'longitude'
# Replace the example values with your desired coordinates
target_lat = 45.5
target_lon = -73.5

# Use .sel() with the method='nearest' argument to find the value at the closest grid point
lsm_at_point = ds['lsm'].sel(latitude=target_lat, longitude=target_lon, method='nearest')

print(f"LSM value at nearest grid point to Lat {target_lat}, Lon {target_lon}:")
print(lsm_at_point.item())

LSM value at nearest grid point to Lat 45.5, Lon -73.5:
1.0


In [11]:
import numpy as np

# Select the lsm DataArray
lsm_data = ds['lsm']

# Get the unique values from the underlying NumPy array
unique_lsm_raw = np.unique(lsm_data.values)

print("--- Raw Unique LSM Values (First 20) ---")
print(unique_lsm_raw[:20])
print(f"\nTotal Number of Unique Values (Raw): {len(unique_lsm_raw)}")

--- Raw Unique LSM Values (First 20) ---
[0.00000000e+00 1.52594876e-05 3.05189752e-05 4.57784628e-05
 6.10379503e-05 7.62974379e-05 9.15569255e-05 1.06816413e-04
 1.22075901e-04 1.37335388e-04 1.52594876e-04 1.67854363e-04
 1.83113851e-04 1.98373339e-04 2.13632826e-04 2.28892314e-04
 2.44151801e-04 2.59411289e-04 2.74670777e-04 2.89930264e-04]

Total Number of Unique Values (Raw): 61357


In [16]:
# Assuming 'lsm_data' is the xarray DataArray selected in the preceding code
# and 'np' is imported as numpy

# Total number of values in lsm_data
total_values = lsm_data.size

# Number of 1s (land)
# We can use the sum() method on the boolean mask (lsm_data == 1)
number_of_ones = (lsm_data == 1).sum().item()

# Number of 0s (sea/ocean)
# We can use the sum() method on the boolean mask (lsm_data == 0)
number_of_zeros = (lsm_data == 0).sum().item()

print(f"\n--- LSM Value Counts ---")
print(f"Total number of values in lsm_data: {total_values}")
print(f"Number of '1's (Land): {number_of_ones}")
print(f"Number of '0's (Sea/Ocean): {number_of_zeros}")
print(f"Sum of '1's and '0's: {number_of_ones + number_of_zeros}")
print(f"Number of all other values: {total_values - (number_of_ones + number_of_zeros)}")



--- LSM Value Counts ---
Total number of values in lsm_data: 6483600
Number of '1's (Land): 1493026
Number of '0's (Sea/Ocean): 4161838
Sum of '1's and '0's: 5654864
Number of all other values: 828736


In [18]:
ocean_mask = ds["lsm"].values == 0
print(ocean_mask, ocean_mask.shape)

[[[ True  True  True ...  True  True  True]
  [ True  True  True ...  True  True  True]
  [ True  True  True ...  True  True  True]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]] (1, 1801, 3600)


In [19]:
ocean_ds = xr.open_dataset(r"D:\Temp\ERA5 Data\2025-02-01-static.nc")
ocean_ds

<xarray.Dataset> Size: 12MB
Dimensions:     (valid_time: 1, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 2025-02-01
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
    expver      <U4 16B ...
Data variables:
    z           (valid_time, latitude, longitude) float32 4MB ...
    lsm         (valid_time, latitude, longitude) float32 4MB ...
    slt         (valid_time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-11-14T14:29 GRIB to CDM+CF via cfgrib-0.9.1...